# Whisper-small LoRA 3×3 조합 선별 실험 — Google Colab

양성 2시간 + 음성(陰性) 2시간의 동일한 subset에 전처리 3종과 학습 파라미터 3종을 조합해 총 9개 실험을 실행합니다. 각 조합은 100 step으로 빠르게 비교하고, 모든 조합의 결과와 LoRA 모델을 저장합니다.

Run all로 패키지 설치 → Drive 연결 → Drive 데이터 직접 사용 → 30초 청킹(10% overlap) → 9개 조합 순차 학습을 진행합니다. 학습 중단 시 5단계 체크포인트에서 재개합니다.

In [2]:
# Run all의 첫 번째 셀: 패키지 설치 → Drive 연결 → Drive 데이터 직접 사용
import subprocess, sys
# 기본 Colab 이미지의 구버전 torchao가 최신 PEFT의 선택적 LoRA dispatcher와 충돌하므로 제거합니다.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'av', 'transformers', 'datasets', 'peft', 'accelerate', 'evaluate', 'jiwer', 'PyYAML', 'soundfile'], check=True)

from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True, timeout_ms=120000)
except Exception as error:
    raise RuntimeError('Google Drive 인증에 실패했습니다. 런타임을 다시 시작하고, Colab을 직접 연 뒤 같은 Google 계정으로 재실행하세요. 브라우저의 팝업/서드파티 쿠키 차단도 해제해야 합니다.') from error

from pathlib import Path
import hashlib, json, math, random, sys, wave, gc, shutil, zipfile
import numpy as np
import torch

PROJECT_ROOT = Path('/content/drive/MyDrive')
SAMPLE_FOLDER_CANDIDATES = [
    Path('/content/drive/MyDrive/colab/stt tuning data'),
    Path('/content/drive/MyDrive/stt tuning data'),
]
SAMPLE_ARCHIVE = Path('/content/drive/MyDrive/prepared_stt_data.zip')
source_root = next((root for root in SAMPLE_FOLDER_CANDIDATES if (root / 'manifest.jsonl').exists()), None)
if source_root is not None:
    # 대용량 데이터를 복사하지 않고 Google Drive에서 직접 읽습니다.
    PREPARED_ROOT = source_root
    print('Drive의 stt tuning data 폴더를 직접 사용합니다. 별도 복사하지 않습니다.')
else:
    LOCAL_ROOT = Path('/content/whisper_sample_upload')
    if LOCAL_ROOT.exists() and not (LOCAL_ROOT / 'manifest.jsonl').exists():
        shutil.rmtree(LOCAL_ROOT)
    if not LOCAL_ROOT.exists():
        if not SAMPLE_ARCHIVE.exists():
            raise FileNotFoundError('Drive의 colab/stt tuning data 또는 stt tuning data 폴더에 manifest.jsonl을 준비하세요.')
        print('Drive의 prepared_stt_data.zip을 Colab 로컬 영역에 압축 해제합니다.')
        LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SAMPLE_ARCHIVE) as archive:
            archive.extractall(LOCAL_ROOT)
        nested = LOCAL_ROOT / 'prepared_stt_data'
        if nested.exists() and (nested / 'manifest.jsonl').exists():
            for item in nested.iterdir():
                target = LOCAL_ROOT / item.name
                if target.exists():
                    shutil.rmtree(target) if target.is_dir() else target.unlink()
                shutil.move(str(item), str(target))
            nested.rmdir()
    PREPARED_ROOT = LOCAL_ROOT
if not (PREPARED_ROOT / 'manifest.jsonl').exists():
    raise FileNotFoundError('stt tuning data 안에 manifest.jsonl이 필요합니다.')
POSITIVE_ROOT = PREPARED_ROOT / 'positive'
NEGATIVE_ROOT = PREPARED_ROOT / 'negative'
if not (POSITIVE_ROOT / 'audio').exists() or not (NEGATIVE_ROOT / 'audio').exists():
    raise FileNotFoundError('stt tuning data 안에 positive/audio와 negative/audio 폴더가 필요합니다.')
if not (PROJECT_ROOT / 'tools').exists():
    raise FileNotFoundError('Google Drive의 MyDrive/proj1-e 폴더에 프로젝트 tools 폴더를 업로드하세요.')
if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임을 GPU로 변경한 뒤 다시 실행하세요.')
sys.path.insert(0, str(PROJECT_ROOT))

from tools.prepare_stt_dataset import SAMPLE_RATE, decode_audio
from tools.train_whisper_lora import WhisperDataCollator, prepare_features
print('GPU:', torch.cuda.get_device_name(0))
print('Drive 데이터 직접 사용·경로 확인 완료:', PREPARED_ROOT)


Mounted at /content/drive
Drive의 stt tuning data 폴더를 직접 사용합니다. 별도 복사하지 않습니다.
GPU: Tesla T4
Drive 데이터 직접 사용·경로 확인 완료: /content/drive/MyDrive/colab/stt tuning data


In [3]:
SEED = 42
TARGET_HOURS_PER_LABEL = 2.0
MAX_AUDIO_SECONDS = 30.0
CHUNK_OVERLAP = 0.10
MIN_CHUNK_SECONDS = 10.0
TRAIN_RATIO, VALIDATION_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

PREPROCESSING = {
    'P0_baseline': dict(speed=(1.0, 1.0), rms=False, gain=(0.0, 0.0), noise_probability=0.0, snr=(30.0, 30.0)),
    'P1_balanced': dict(speed=(0.9, 1.1), rms=True, gain=(-3.0, 3.0), noise_probability=0.20, snr=(15.0, 30.0)),
    'P2_robust': dict(speed=(0.9, 1.1), rms=True, gain=(-6.0, 6.0), noise_probability=0.35, snr=(10.0, 25.0)),
}

# 1차 실험은 T1 안정형 설정만 사용합니다.
TRAINING = {
    'T1_screening': dict(lr=2e-5, lora_r=8, lora_alpha=16, dropout=0.05, max_steps=100),
    'T2_screening': dict(lr=5e-5, lora_r=8, lora_alpha=16, dropout=0.05, max_steps=100),
    'T3_screening': dict(lr=1e-4, lora_r=16, lora_alpha=32, dropout=0.10, max_steps=100),
}

BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
WARMUP_STEPS = 100
LOGGING_STEPS = 25
SAVE_STEPS = 250
OUTPUT_ROOT = PROJECT_ROOT / 'models' / 'whisper-lora-screening-3x3'


In [4]:
def wav_duration(path):
    with wave.open(str(path), 'rb') as handle:
        return handle.getnframes() / handle.getframerate()

def positive_path(record, field):
    parts = Path(record[field]).parts
    try: index = parts.index('data')
    except ValueError: return Path(record[field])
    return POSITIVE_ROOT.joinpath(*parts[index + 1:])

def sample_path(record, field):
    # Colab은 Linux지만 manifest 경로는 Windows 형식일 수 있으므로 구분자를 직접 처리합니다.
    raw = str(record[field]).replace('\\', '/')
    parts = [part for part in raw.split('/') if part]
    marker_index = next((i for i, part in enumerate(parts) if part.lower() == 'prepared_stt_data'), None)
    if marker_index is not None:
        return PREPARED_ROOT.joinpath(*parts[marker_index + 1:])
    candidate = Path(raw)
    if candidate.is_absolute() and candidate.exists():
        return candidate
    return PREPARED_ROOT / candidate.name

def source_pairs(record):
    return [(sample_path(record, 'audio'), sample_path(record, 'text'), record['id'], record['id'])]

def split_transcript(transcript, start_seconds, end_seconds, total_seconds):
    transcript = transcript.strip()
    if not transcript:
        return ''
    left = int(len(transcript) * start_seconds / total_seconds)
    right = int(len(transcript) * end_seconds / total_seconds)
    if left > 0:
        while left < len(transcript) and not transcript[left].isspace():
            left += 1
    if right < len(transcript):
        while right > left and not transcript[right - 1].isspace():
            right -= 1
    chunk = transcript[left:right].strip()
    return chunk or transcript

def build_pairs():
    records = [json.loads(line) for line in (PREPARED_ROOT / 'manifest.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    pairs, skipped = [], []
    chunk_step = MAX_AUDIO_SECONDS * (1.0 - CHUNK_OVERLAP)
    for record in records:
        for audio, text, pair_id, call_id in source_pairs(record):
            if not audio.exists() or not text.exists():
                skipped.append((pair_id, 'missing_pair'))
                continue
            try:
                duration = wav_duration(audio) if audio.suffix.lower() == '.wav' else len(decode_audio(audio)) / SAMPLE_RATE
                transcript = text.read_text(encoding='utf-8-sig').strip()
            except Exception as error:
                skipped.append((pair_id, type(error).__name__))
                continue
            if duration <= MAX_AUDIO_SECONDS:
                pairs.append(dict(id=pair_id, call_id=call_id, label=record['label'], audio=audio, text=text, transcript=transcript, duration=duration, start_seconds=0.0, end_seconds=duration))
                continue
            chunk_index = 0
            start = 0.0
            while start < duration:
                end = min(start + MAX_AUDIO_SECONDS, duration)
                chunk_duration = end - start
                if chunk_duration >= MIN_CHUNK_SECONDS:
                    pairs.append(dict(id=f'{pair_id}__chunk{chunk_index:04d}', call_id=call_id, label=record['label'], audio=audio, text=text, transcript=split_transcript(transcript, start, end, duration), duration=chunk_duration, start_seconds=start, end_seconds=end))
                    chunk_index += 1
                start += chunk_step
    return pairs, skipped

pairs, skipped = build_pairs()
print('usable chunks:', len(pairs), 'skipped source files:', len(skipped))
print('reasons:', {r: sum(1 for _, x in skipped if x == r) for r in sorted({x for _, x in skipped})})
if not pairs:
    raise RuntimeError('usable chunks가 0개입니다. Drive의 stt tuning data 경로와 manifest/audio/labels 구조를 확인하세요.')


usable chunks: 5597 skipped source files: 0
reasons: {}


In [5]:
def choose(items, hours, seed):
    remaining = hours * 3600; pool = list(items); random.Random(seed).shuffle(pool); chosen = []
    for item in pool:
        if item['duration'] <= remaining: chosen.append(item); remaining -= item['duration']
        if remaining <= 1: break
    return chosen
selected = []
for label, offset in [('positive', 0), ('negative', 1)]:
    chosen = choose([x for x in pairs if x['label'] == label], TARGET_HOURS_PER_LABEL, SEED + offset)
    selected += chosen
    if not chosen:
        raise RuntimeError(f'{label} 샘플이 0개입니다. usable chunks와 파일 경로를 확인하세요.')
    print(label, len(chosen), round(sum(x['duration'] for x in chosen) / 3600, 3), 'hours')
groups = {}
for item in selected: groups.setdefault(item['call_id'], []).append(item)
group_list = list(groups.values()); random.Random(SEED).shuffle(group_list)
splits = {'train': [], 'validation': [], 'test': []}; totals = {x: 0.0 for x in splits}; ratios = {'train': TRAIN_RATIO, 'validation': VALIDATION_RATIO, 'test': TEST_RATIO}
for group in group_list:
    name = min(splits, key=lambda x: totals[x] / ratios[x]); splits[name].extend(group); totals[name] += sum(x['duration'] for x in group)
print({x: {'pairs': len(splits[x]), 'hours': round(totals[x] / 3600, 3)} for x in splits})
if not splits['train'] or not splits['validation']:
    raise RuntimeError('train 또는 validation split이 비어 있습니다. 샘플 수와 분할 비율을 확인하세요.')
(OUTPUT_ROOT / 'selection').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'selection' / 'pairs.json').write_text(json.dumps({'splits': {k: [{**x, 'audio': str(x['audio']), 'text': str(x['text'])} for x in v] for k, v in splits.items()}}, ensure_ascii=False, indent=2), encoding='utf-8')

positive 245 1.999 hours
negative 246 1.997 hours
{'train': {'pairs': 342, 'hours': 2.782}, 'validation': {'pairs': 74, 'hours': 0.61}, 'test': {'pairs': 75, 'hours': 0.605}}


390531

In [6]:
from functools import lru_cache

def rng_for(identifier, seed): return np.random.default_rng(int.from_bytes(hashlib.sha256(f'{seed}:{identifier}'.encode()).digest()[:8], 'little'))
def rms_db(x): return 20 * math.log10(max(float(np.sqrt(np.mean(x*x))), 1e-8))
def preprocess(x, identifier, cfg):
    rng = rng_for(identifier, SEED); lo, hi = cfg['speed']; factor = float(rng.uniform(lo, hi))
    if factor != 1.0: x = np.interp(np.arange(0, len(x), factor), np.arange(len(x)), x).astype('float32')
    if cfg['rms']: x = x * 10 ** ((-20.0 - rms_db(x)) / 20)
    gain_lo, gain_hi = cfg['gain']; x = x * 10 ** (float(rng.uniform(gain_lo, gain_hi)) / 20)
    if rng.random() < cfg['noise_probability']:
        signal = max(float(np.sqrt(np.mean(x*x))), 1e-8); snr = float(rng.uniform(*cfg['snr']))
        x = x + rng.normal(0, signal / 10 ** (snr / 20), len(x)).astype('float32')
    return np.clip(x, -1, 1).astype('float32')

@lru_cache(maxsize=128)
def decode_cached(path_string):
    return decode_audio(Path(path_string))

class LocalDataset:
    def __init__(self, items, processor, cfg): self.items, self.processor, self.cfg = items, processor, cfg
    def __len__(self): return len(self.items)
    def __getitem__(self, index):
        item = self.items[index]
        full_audio = decode_cached(str(item['audio']))
        start = int(item['start_seconds'] * SAMPLE_RATE)
        end = min(int(item['end_seconds'] * SAMPLE_RATE), len(full_audio))
        audio = preprocess(full_audio[start:end].copy(), item['id'], self.cfg)
        return prepare_features({'audio': {'array': audio, 'sampling_rate': SAMPLE_RATE}, 'transcript': item['transcript']}, self.processor)


In [7]:
from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration, WhisperProcessor, Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint
import time

class StepProgressCallback(TrainerCallback):
    def __init__(self, run_name, stage_index, stage_count, total_steps):
        self.run_name = run_name
        self.stage_index = stage_index
        self.stage_count = stage_count
        self.total_steps = total_steps
        self.started_at = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        print(f'[{self.run_name}] STAGE {self.stage_index}/{self.stage_count} START: step {state.global_step}/{self.total_steps}')

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or not state.is_world_process_zero:
            return
        step = int(state.global_step)
        total = max(int(self.total_steps), 1)
        elapsed = time.time() - self.started_at if self.started_at else 0.0
        rate = step / elapsed if elapsed > 0 else 0.0
        remaining = (total - step) / rate if rate > 0 else 0.0
        progress = 100.0 * step / total
        loss = logs.get('loss', '-')
        lr = logs.get('learning_rate', '-')
        print(f'[{self.run_name}] STAGE {self.stage_index}/{self.stage_count} | step {step}/{total} ({progress:5.1f}%) | loss={loss} | lr={lr} | elapsed={elapsed/60:.1f}m | ETA={remaining/60:.1f}m')

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.started_at if self.started_at else 0.0
        print(f'[{self.run_name}] STAGE {self.stage_index}/{self.stage_count} END: step {state.global_step}/{self.total_steps} | elapsed={elapsed/60:.1f}m')

STAGE_COUNT = 5  # screening: 20 step씩 5단계

for p_name, p_cfg in PREPROCESSING.items():
    for t_name, t_cfg in TRAINING.items():
        run_name = f'{p_name}__{t_name}'
        run_dir = OUTPUT_ROOT / run_name
        run_dir.mkdir(parents=True, exist_ok=True)
        done_marker = run_dir / '_SUCCESS'
        if done_marker.exists():
            print('SKIP', run_name, '(already completed)')
            continue

        total_steps = int(t_cfg['max_steps'])
        stage_step = max(1, total_steps // STAGE_COUNT)
        stage_ends = [min(total_steps, stage_step * i) for i in range(1, STAGE_COUNT)] + [total_steps]
        print(f'START {run_name}: {STAGE_COUNT} stages, total {total_steps} steps')

        for stage_index, stage_end in enumerate(stage_ends, start=1):
            stage_marker = run_dir / f'_STAGE_{stage_index:02d}_SUCCESS'
            if stage_marker.exists():
                print(f'SKIP {run_name} STAGE {stage_index}/{STAGE_COUNT} (already completed)')
                continue

            processor = WhisperProcessor.from_pretrained('openai/whisper-small', language='ko', task='transcribe')
            model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')
            model.generation_config.language, model.generation_config.task = 'ko', 'transcribe'
            model.generation_config.forced_decoder_ids = None
            model.config.use_cache = False
            model = get_peft_model(model, LoraConfig(r=t_cfg['lora_r'], lora_alpha=t_cfg['lora_alpha'], lora_dropout=t_cfg['dropout'], target_modules=['q_proj', 'v_proj'], bias='none', task_type=None))
            if hasattr(model, 'enable_input_require_grads'):
                model.enable_input_require_grads()
            for name, parameter in model.named_parameters():
                if '.encoder.' in name or name.startswith('encoder.'):
                    parameter.requires_grad = False

            args_values = dict(
                output_dir=str(run_dir),
                per_device_train_batch_size=BATCH_SIZE,
                per_device_eval_batch_size=BATCH_SIZE,
                gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
                learning_rate=t_cfg['lr'],
                warmup_steps=WARMUP_STEPS,
                logging_steps=LOGGING_STEPS,
                save_steps=min(SAVE_STEPS, stage_step),
                eval_steps=min(SAVE_STEPS, stage_step),
                save_total_limit=2,
                fp16=True,
                gradient_checkpointing=True,
                remove_unused_columns=False,
                report_to=[],
                predict_with_generate=False,
                disable_tqdm=False,
                evaluation_strategy='steps',
                save_strategy='steps',
                max_steps=stage_end,
                num_train_epochs=1,
            )
            try:
                training_args = Seq2SeqTrainingArguments(**args_values)
            except TypeError:
                args_values['eval_strategy'] = args_values.pop('evaluation_strategy')
                training_args = Seq2SeqTrainingArguments(**args_values)

            trainer = Seq2SeqTrainer(
                args=training_args,
                model=model,
                train_dataset=LocalDataset(splits['train'], processor, p_cfg),
                eval_dataset=LocalDataset(splits['validation'], processor, p_cfg),
                data_collator=WhisperDataCollator(processor),
                callbacks=[StepProgressCallback(run_name, stage_index, STAGE_COUNT, total_steps)],
            )
            trainer.train(resume_from_checkpoint=get_last_checkpoint(str(run_dir)))
            trainer.save_model(run_dir)
            processor.save_pretrained(run_dir)
            trainer.save_state()
            (run_dir / 'run_config.json').write_text(json.dumps({'preprocessing': p_cfg, 'training': t_cfg, 'stage_count': STAGE_COUNT, 'stage_end_steps': stage_ends, 'selection_hours': {k: totals[k] / 3600 for k in totals}}, ensure_ascii=False, indent=2), encoding='utf-8')
            stage_marker.write_text('completed\n', encoding='utf-8')
            del trainer, model, processor
            gc.collect()
            torch.cuda.empty_cache()
            print(f'DONE {run_name} STAGE {stage_index}/{STAGE_COUNT}')

        done_marker.write_text('completed\n', encoding='utf-8')
        print('DONE', run_name)


START P0_baseline__T1_screening: 5 stages, total 100 steps


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[P0_baseline__T1_screening] STAGE 1/5 START: step 0/100


Step,Training Loss,Validation Loss
20,No log,3.046512


...

[P2_robust__T3_screening] STAGE 4/5 | step 75/100 ( 75.0%) | loss=16.952930704752603 | lr=7.4e-05 | elapsed=1.3m | ETA=0.4m
[P2_robust__T3_screening] STAGE 4/5 | step 80/100 ( 80.0%) | loss=- | lr=- | elapsed=2.0m | ETA=0.5m
[P2_robust__T3_screening] STAGE 4/5 | step 80/100 ( 80.0%) | loss=- | lr=- | elapsed=2.1m | ETA=0.5m
[P2_robust__T3_screening] STAGE 4/5 END: step 80/100 | elapsed=2.1m
DONE P2_robust__T3_screening STAGE 4/5


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[P2_robust__T3_screening] STAGE 5/5 START: step 80/100


Step,Training Loss,Validation Loss
100,16.201390,2.546027


[P2_robust__T3_screening] STAGE 5/5 | step 100/100 (100.0%) | loss=16.201390075683594 | lr=9.900000000000001e-05 | elapsed=1.7m | ETA=0.0m
[P2_robust__T3_screening] STAGE 5/5 | step 100/100 (100.0%) | loss=- | lr=- | elapsed=2.1m | ETA=0.0m
[P2_robust__T3_screening] STAGE 5/5 | step 100/100 (100.0%) | loss=- | lr=- | elapsed=2.1m | ETA=0.0m
[P2_robust__T3_screening] STAGE 5/5 END: step 100/100 | elapsed=2.1m
DONE P2_robust__T3_screening STAGE 5/5
DONE P2_robust__T3_screening
